# PRIDE — Notebook 2: Real-Data Calibrated TV Proxy

Fills **Table V** of the revised manuscript.

Evaluates PRIDE on three real-world datasets (UCI Adult, UCI HAR, UCI Air Quality)
and one synthetic Gaussian baseline.  Reports the classifier-based TV proxy:

    δ̂_TV^cls = 2 × (AUC − 0.5)

where AUC is from a logistic-regression two-sample test that tries to
distinguish real records from PRIDE-generated decoys.

**All datasets auto-download** when the relevant cells are executed.
No manual file placement is required.

Dataset changes vs. original submission:
  - Intel Berkeley Lab sensor data REMOVED (no auto-download)
  - UCI Air Quality (id=360, CO sensor) ADDED as IoT-relevant replacement

## Cell 1 — Installs (run once)

`ucimlrepo` is the official UCI ML Repository Python client.
If it is already installed this cell does nothing.

In [6]:
import subprocess, sys
def _install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

try:
    import ucimlrepo  # noqa: F401
except ImportError:
    print("Installing ucimlrepo ...")
    _install('ucimlrepo')

try:
    import sklearn  # noqa: F401
except ImportError:
    print("Installing scikit-learn ...")
    _install('scikit-learn')

print("Dependencies OK.")

Dependencies OK.


## Cell 2 — Imports

In [9]:
import os, sys
import numpy as np
import pandas as pd

# __file__ is undefined in Jupyter; fall back to cwd (works in both contexts)
sys.path.insert(0, str(__import__('pathlib').Path(
    globals().get('__file__', '.')).resolve().parent))

from pride_core import (
    PRIDE, PRIDEParams, GaussianSampler, EmpiricalSampler,
    calibrated_tv_classifier, empirical_tv_binned,
    load_adult, load_har, load_airquality,
)

SEED = 40
N_RECORDS = 2_000    # real records per dataset (enough for AUC stability)
K = 2                # decoys per real record (paper's default)
N_THR = 3            # threshold n

rng = np.random.default_rng(SEED)
print(f"Config: N_RECORDS={N_RECORDS}, K={K}, n={N_THR}")

Config: N_RECORDS=2000, K=2, n=3


## Cell 3 — Dataset loaders

Each dataset is loaded once and cached to disk under `./data/`.
Subsequent runs use the cache and skip the download.

In [12]:
print("Loading UCI Adult ...")
adult_vals = load_adult()
print(f"  Adult: {len(adult_vals)} records, "
      f"mean={adult_vals.mean():.1f}, std={adult_vals.std():.1f}")

print("\nLoading UCI HAR ...")
har_vals, har_labels = load_har()
print(f"  HAR PCA-1: {len(har_vals)} records, "
      f"mean={har_vals.mean():.3f}, std={har_vals.std():.3f}")

print("\nLoading UCI Air Quality ...")
air_vals = load_airquality()
print(f"  Air Quality CO: {len(air_vals)} records, "
      f"mean={air_vals.mean():.3f}, std={air_vals.std():.3f}")

print("\nAll datasets loaded.")

Loading UCI Adult ...
  [cached] Adult age: N=48842
  Adult: 48842 records, mean=38.6, std=13.7

Loading UCI HAR ...
  [cached] HAR: N=10299
  HAR PCA-1: 10299 records, mean=-0.000, std=5.809

Loading UCI Air Quality ...
Fetching UCI Air Quality via ucimlrepo (id=360) ...
  ucimlrepo: N=7674, mean=2.153, std=1.453
  Air Quality CO: 7674 records, mean=2.153, std=1.453

All datasets loaded.


## Cell 4 — Decoy generator factory

Each dataset gets a decoy generator trained on its distribution.

- **Adult**: marginal empirical sampler (resample from the training split)
- **HAR**: per-class Gaussian mixture (approximate; 6 activity classes)
- **Air Quality**: KDE-style empirical sampler with small Gaussian jitter
- **Synthetic Gaussian**: exact P (sanity-check row — AUC should be ≈ 0.5)

In [15]:
def make_adult_sampler(vals: np.ndarray, seed: int) -> EmpiricalSampler:
    """Marginal sampler: resample from observed distribution."""
    return EmpiricalSampler(vals, jitter=0.0, seed=seed)


def make_har_sampler(vals: np.ndarray, labels: np.ndarray, seed: int):
    """
    Per-class Gaussian mixture: fit N(mu_c, sigma_c) per activity class,
    sample class proportionally to its size, then sample from that Gaussian.
    """
    classes = np.unique(labels)
    class_params = {}
    for c in classes:
        v = vals[labels == c]
        class_params[c] = (float(v.mean()), float(v.std()) + 1e-9)
    class_counts = np.array([np.sum(labels == c) for c in classes], dtype=float)
    weights = class_counts / class_counts.sum()
    rng_local = np.random.default_rng(seed)

    def _sampler():
        c = rng_local.choice(classes, p=weights)
        mu, sigma = class_params[c]
        return float(rng_local.normal(mu, sigma))

    return _sampler


def make_air_sampler(vals: np.ndarray, seed: int) -> EmpiricalSampler:
    """KDE-style: empirical resampling with small Gaussian jitter."""
    jitter = float(vals.std()) * 0.05   # 5% bandwidth
    return EmpiricalSampler(vals, jitter=jitter, seed=seed)


def make_gaussian_sampler(seed: int) -> GaussianSampler:
    return GaussianSampler(mean=1000.0, std=100.0, seed=seed)

print("Decoy generator factory defined.")

Decoy generator factory defined.


## Cell 5 — Calibrated TV computation

For each dataset:
  1. Sample N_RECORDS real values from the dataset.
  2. Encrypt them with PRIDE (K decoys per record).
  3. Decrypt one decoy per ciphertext (simulates what the adversary sees).
  4. Run the classifier-based two-sample test (Section V.B of the paper).

In [18]:
def run_tv_experiment(name: str, real_vals: np.ndarray,
                       decoy_sampler_fn, N: int, K: int, n: int,
                       seed: int) -> dict:
    """
    Encrypt N real records with K PRIDE decoys each, then measure the
    classifier-based TV proxy between the real values and sampled decoys.
    """
    rng_local = np.random.default_rng(seed)
    # Sample N real records from the provided values
    idx = rng_local.choice(len(real_vals), size=N, replace=(len(real_vals) < N))
    real_sample = real_vals[idx]

    # PRIDE encryption to build the ciphertext + key set
    params = PRIDEParams(n=n, K=K, init_shares=n + 2)
    system = PRIDE(params, seed=seed)
    decoy_sampler = decoy_sampler_fn(seed + 1)
    ct = system.setup()

    decoy_revealed = []
    for m in real_sample:
        m_int = int(round(float(m))) % params.p
        sks = system.enc(ct, m_int, decoy_sampler)
        # Reveal one randomly chosen decoy (not the real key sk_0)
        dk = sks[int(rng_local.integers(1, len(sks)))]
        decoy_revealed.append(system.dec(ct, dk))

    decoy_arr = np.array(decoy_revealed, dtype=float)
    # Bring back to original scale (mod p wraps some values — clip extremes)
    p = params.p
    decoy_arr = np.where(decoy_arr > p // 2, decoy_arr - p, decoy_arr)

    auc_delta = calibrated_tv_classifier(real_sample, decoy_arr, seed=seed)
    tv_binned  = empirical_tv_binned(real_sample, decoy_arr)

    auc_val = (auc_delta / 2.0) + 0.5
    return {
        'Dataset': name,
        'N': N,
        'AUC': round(auc_val, 4),
        'delta_TV_cls': round(auc_delta, 4),
        'delta_TV_binned': round(tv_binned, 4),
    }


print("TV experiment function defined.")

TV experiment function defined.


## Cell 6 — Run all four rows

This cell takes roughly 1–3 minutes on a standard laptop.

In [21]:
rows = []

print("Running: Synthetic Gaussian (sanity check) ...")
r = run_tv_experiment(
    'Synthetic Gaussian (orig.)',
    np.random.default_rng(SEED).normal(1000, 100, 100_000).astype(float),
    lambda s: GaussianSampler(mean=1000.0, std=100.0, seed=s),
    N=N_RECORDS, K=K, n=N_THR, seed=SEED
)
rows.append(r)
print(f"  AUC={r['AUC']:.4f}  δ̂_TV^cls={r['delta_TV_cls']:.4f}  (expect ≈ 0.500 / 0.000)")

print("\nRunning: UCI Adult ...")
r = run_tv_experiment(
    'Adult (age)',
    adult_vals,
    lambda s: make_adult_sampler(adult_vals, s),
    N=N_RECORDS, K=K, n=N_THR, seed=SEED
)
rows.append(r)
print(f"  AUC={r['AUC']:.4f}  δ̂_TV^cls={r['delta_TV_cls']:.4f}")

print("\nRunning: UCI HAR ...")
r = run_tv_experiment(
    'HAR (PCA-1)',
    har_vals,
    lambda s: make_har_sampler(har_vals, har_labels, s),
    N=N_RECORDS, K=K, n=N_THR, seed=SEED
)
rows.append(r)
print(f"  AUC={r['AUC']:.4f}  δ̂_TV^cls={r['delta_TV_cls']:.4f}")

print("\nRunning: UCI Air Quality (CO sensor) ...")
r = run_tv_experiment(
    'Air Quality (CO)',
    air_vals,
    lambda s: make_air_sampler(air_vals, s),
    N=N_RECORDS, K=K, n=N_THR, seed=SEED
)
rows.append(r)
print(f"  AUC={r['AUC']:.4f}  δ̂_TV^cls={r['delta_TV_cls']:.4f}")

print("\nAll rows complete.")

Running: Synthetic Gaussian (sanity check) ...
  AUC=0.4974  δ̂_TV^cls=-0.0052  (expect ≈ 0.500 / 0.000)

Running: UCI Adult ...
  AUC=0.5003  δ̂_TV^cls=0.0006

Running: UCI HAR ...
  AUC=0.6547  δ̂_TV^cls=0.3093

Running: UCI Air Quality (CO sensor) ...
  AUC=0.4978  δ̂_TV^cls=-0.0044

All rows complete.


## Cell 7 — Table V output

In [24]:
df = pd.DataFrame(rows)

print("\n" + "="*72)
print("TABLE V: Calibrated TV proxy — Real-data design (PRIDE)")
print("="*72)
print(f"{'Dataset':<30} {'N':>7} {'Distinguish AUC':>16} {'δ̂_TV^cls':>12}")
print("-"*72)
for _, row in df.iterrows():
    print(f"{row['Dataset']:<30} {row['N']:>7} "
          f"{row['AUC']:>16.4f} {row['delta_TV_cls']:>12.4f}")
print("="*72)

# Sanity check: synthetic Gaussian AUC must be ≈ 0.5
sg = df[df['Dataset'].str.contains('Gaussian')]
if len(sg) and abs(float(sg['AUC'].iloc[0]) - 0.5) < 0.05:
    print("\nSanity check PASSED: Gaussian AUC ≈ 0.500 (expect chance level)")
else:
    print("\nSanity check NOTE: Gaussian AUC deviated from 0.5 — re-check")


TABLE V: Calibrated TV proxy — Real-data design (PRIDE R1)
Dataset                              N  Distinguish AUC    δ̂_TV^cls
------------------------------------------------------------------------
Synthetic Gaussian (orig.)        2000           0.4974      -0.0052
Adult (age)                       2000           0.5003       0.0006
HAR (PCA-1)                       2000           0.6547       0.3093
Air Quality (CO)                  2000           0.4978      -0.0044

Sanity check PASSED: Gaussian AUC ≈ 0.500 (expect chance level)
